### Carregando o modelo

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'Qwen/Qwen2-1.5B-Instruct' #gpt2
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
from transformers.utils import logging

logging.set_verbosity_error()

### Utilizado pipeline para geração de texto

In [ ]:
from transformers import pipeline

gerador = pipeline('text-generation', model=model, tokenizer=tokenizer)
resultado = gerador('Era uma vez', max_length=25)
print(resultado)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=25) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Era uma vez um homem chamado João, que tinha um grande desejo de ser um grande guerreiro. Ele sempre sonhou em combater bravamente e vencer batalhas impressionantes.\n\nUm dia, ele decidiu que era hora de realizar esse sonho. Ele se preparou com todas as suas habilidades e força para enfrentar qualquer adversidade que possa surgir. \n\nEle começou sua jornada com muita coragem e determinação. Ele viajou longe de casa, atravessando vastas terras e enfrentando inúmeros desafios. \n\nApesar do perigo e do estresse, João nunca desistiu. Ele sabia que a realização dos seus sonhos exigia sacrifício e trabalho duro. \n\nFinalmente, após muito tempo, João encontrou seu destino final: um gigante imponente e poderoso. O gigante, ao ver João, o acolheu com grande gentileza e respeito. E assim, João tornou-se um verdadeiro guerreiro, lutando por sua própria causa e conquistando os honores.\n\nPortanto, podemos dizer que, mesmo quando parecem impossíveis, os sonhos podem se tor

In [ ]:
resultado[0]['generated_text']

'Era uma vez um homem chamado João, que tinha um grande desejo de ser um grande guerreiro. Ele sempre sonhou em combater bravamente e vencer batalhas impressionantes.\n\nUm dia, ele decidiu que era hora de realizar esse sonho. Ele se preparou com todas as suas habilidades e força para enfrentar qualquer adversidade que possa surgir. \n\nEle começou sua jornada com muita coragem e determinação. Ele viajou longe de casa, atravessando vastas terras e enfrentando inúmeros desafios. \n\nApesar do perigo e do estresse, João nunca desistiu. Ele sabia que a realização dos seus sonhos exigia sacrifício e trabalho duro. \n\nFinalmente, após muito tempo, João encontrou seu destino final: um gigante imponente e poderoso. O gigante, ao ver João, o acolheu com grande gentileza e respeito. E assim, João tornou-se um verdadeiro guerreiro, lutando por sua própria causa e conquistando os honores.\n\nPortanto, podemos dizer que, mesmo quando parecem impossíveis, os sonhos podem se tornar realidade com'

### Gerando texto manualmente

In [ ]:
input_ids = tokenizer.encode('Era uma vez', return_tensors='pt')
input_ids

tensor([[   36,   956, 10608, 20563]])

In [ ]:
output = model.generate(input_ids, max_length=25)
output

tensor([[   36,   956, 10608, 20563,  4443,   312,    72, 33750, 27030,   384,
         27030,    13, 25949, 11385, 70483,  5249,  4154, 19345, 22218,   783,
          1021,   384, 27525, 20009,   282]])

In [ ]:
resultado = tokenizer.decode(output[0], skip_special_tokens=True)
print(resultado)

Era uma vez um rei muito cruel e cruel. Ele era conhecido por sua crueldade e pelo seu f


### Criando uma estrutura de conversação

In [ ]:
messages = [{'role':'user', 'content':'Olá'}]
text = tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors='pt')
inputs


{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  42719,   1953, 151645,    198,
         151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
outputs = model.generate(inputs.input_ids, max_length=25, do_sample=True)
resultado = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(resultado)

system
You are a helpful assistant.
user
Olá
assistant
Olá! Como


#### Criando um pequeno loop de conversa

In [ ]:
mensagens = []
for i in range(3):
    messages.append({'role':'user', 'content':input('Fale algo com o assitente: ')})
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt')
    outputs = model.generate(inputs.input_ids, max_length=500, do_sample=True)
    resultado = tokenizer.decode(outputs[0], skip_special_tokens=True)
    resposta_assist = resultado.split('assistant')[-1].strip('/n')
    print(resposta_assist)
    messages.append({'role':'assistant', 'content':resposta_assist})

Fale algo com o assitente: E ai tudo bem?

Olá! Estou bem, obrigado. Como posso ajudar você hoje?
Fale algo com o assitente: Que dia da semana é hoje?

Desculpe, como sou um assistente virtual de inteligência artificial, não tenho a capacidade de fornecer informações em tempo real ou horários. Por favor, verifique sua agenda ou calendário para obter essa informação.
Fale algo com o assitente: Quanto é 2+2?

O resultado de 2 + 2 é 4.
